# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [ ]:
%pip -q install duckdb huggingface_hub


In [1]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [2]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [3]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [4]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_e547b89c05043229,content_6b80dfab2e0ffa2e,1110.0,955.0,12.0,7.543789
1,client_e547b89c05043229,content_d7bb60ec9a42c11a,3735.0,3338.0,33.0,5.446636
2,client_e547b89c05043229,content_401dcc5cd616e3dd,181.0,130.0,0.0,6.874167
3,client_e547b89c05043229,content_18d95bd7890430ed,151.0,340.0,0.0,33.665367
4,client_e547b89c05043229,content_56f46c55f0348ab4,392.0,531.0,3.0,12.995100


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [5]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_e547b89c05043229,content_6b80dfab2e0ffa2e,1110.0,955.0,12.0,7.543789,1.0,0.022750,0.957216,59.0,59.0,1.000000
1,client_e547b89c05043229,content_d7bb60ec9a42c11a,3735.0,3338.0,33.0,5.446636,14.0,0.017946,0.932994,84.0,462.0,0.181818
2,client_e547b89c05043229,content_401dcc5cd616e3dd,181.0,130.0,0.0,6.874167,3.0,0.162037,0.552469,153.0,185.0,0.827027
3,client_e547b89c05043229,content_18d95bd7890430ed,151.0,340.0,0.0,33.665367,2.0,0.108932,0.820261,52.0,65.0,0.800000
4,client_e547b89c05043229,content_56f46c55f0348ab4,392.0,531.0,3.0,12.995100,5.0,0.163052,0.788332,14.0,65.0,0.215385


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [6]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


base rate (always predict majority): 0.633
              precision    recall  f1-score   support

           0      0.547     0.340     0.419      9389
           1      0.686     0.836     0.754     16162

    accuracy                          0.654     25551
   macro avg      0.616     0.588     0.587     25551
weighted avg      0.635     0.654     0.631     25551



## 1. Signal checks
### Signal 1 - Previous 30-day impressions
imp_prev30 shows only a weak relationship with declining behavior. Decline rates increase slightly across the lower-to-higher impression buckets, but the difference is small and the highest bucket does not continue the monotonic trend.

Signal: imp_prev30

Verdict: MIXED

In [7]:
model_data['imp_prev30'].describe()

,imp_prev30
count,102203.000000
mean,2454.496903
std,7214.444150
min,100.000000
25%,263.000000
50%,646.000000
75%,1982.000000
max,585502.000000


In [8]:
import pandas as pd

buckets = pd.qcut(
    model_data['imp_prev30'],
    q=4,
    labels=['Low','Medium-Low','Medium-High', 'High']
)

signal_check_1 = (
    model_data.assign(imp_prev30_bucket=buckets).
    groupby('imp_prev30_bucket', observed=False).agg(
        m=('is_declining', 'size'),
        declining_rate=('is_declining', 'mean')
    )
    .reset_index()
)
signal_check_1

,imp_prev30_bucket,m,declining_rate
0,Low,25647,0.612703
1,Medium-Low,25480,0.629042
2,Medium-High,25529,0.647616
3,High,25547,0.640819


##

## Signal 2 - Visible Queries

The signal behaves in the opposite direction from the expected relationship: higher visible_queries is associated with a lower declining rate, while lower visible_queries is associated with a higher declining rate.

Signal: visible_queries

Verdict: OPPOSITE

In [9]:
model_data['visible_queries'].describe()

,visible_queries
count,102203.000000
mean,22.754684
std,52.835191
min,1.000000
25%,4.000000
50%,9.000000
75%,23.000000
max,7889.000000


In [10]:
buckets = pd.qcut(
    model_data['visible_queries'],
    q=4,
    labels=['Low', 'Medium-Low', 'Medium-High', 'High']
)

signal_check_2 = (
    model_data.assign(visible_queries_bucket=buckets)
    .groupby('visible_queries_bucket', observed=False)
    .agg(
        n=('is_declining', 'size'),
        declining_rate=('is_declining', 'mean')
    )
    .reset_index()
)

signal_check_2

,visible_queries_bucket,n,declining_rate
0,Low,29217,0.694424
1,Medium-Low,22129,0.630169
2,Medium-High,25677,0.596487
3,High,25180,0.599523


In [13]:
baseline = model_data.copy()

baseline['score'] = pd.cut(
    baseline['visible_queries'],
    bins=[-np.inf, 4, 9, 23, np.inf],
    labels=[3, 2, 1, 0],
    right=True
).astype(int)

baseline['reason_code'] = 'LOW_QUERY_VISIBILITY'

baseline['action_label'] = np.select(
    [
        baseline['score'] == 3,
        baseline['score'] == 2,
        baseline['score'] == 1
    ],
    [
        'QUICK_WIN',
        'REVIEW',
        'MONITOR'
    ],
    default='NO_ACTION'
)

baseline[['content_hash_id', 'score', 'reason_code', 'action_label']].head()

,content_hash_id,score,reason_code,action_label
0,content_6b80dfab2e0ffa2e,3,LOW_QUERY_VISIBILITY,QUICK_WIN
1,content_d7bb60ec9a42c11a,1,LOW_QUERY_VISIBILITY,MONITOR
2,content_401dcc5cd616e3dd,3,LOW_QUERY_VISIBILITY,QUICK_WIN
3,content_18d95bd7890430ed,3,LOW_QUERY_VISIBILITY,QUICK_WIN
4,content_56f46c55f0348ab4,2,LOW_QUERY_VISIBILITY,REVIEW


In [14]:
baseline_queue = (
    baseline[
        ['client_hash_id', 'content_hash_id',
         'score', 'reason_code', 'action_label']
    ]
    .sort_values(
        by=['score', 'content_hash_id'],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

baseline_queue.head(10)

,client_hash_id,content_hash_id,score,reason_code,action_label
0,client_7de9989c909e91a5,content_00000c99413ae2ad,3,LOW_QUERY_VISIBILITY,QUICK_WIN
1,client_9958f0a7ae1df715,content_0002bd310bf01f15,3,LOW_QUERY_VISIBILITY,QUICK_WIN
2,client_fef1a8f436438636,content_00032be2df0005ca,3,LOW_QUERY_VISIBILITY,QUICK_WIN
3,client_73cda7b4e4f265ea,content_00033c286cc93446,3,LOW_QUERY_VISIBILITY,QUICK_WIN
4,client_157ffe4d4a595515,content_00039f4c7a954114,3,LOW_QUERY_VISIBILITY,QUICK_WIN
5,client_73cda7b4e4f265ea,content_0009df3c2b96f209,3,LOW_QUERY_VISIBILITY,QUICK_WIN
6,client_e5c2aa26a8598242,content_000b09c2a4ef4ba8,3,LOW_QUERY_VISIBILITY,QUICK_WIN
7,client_62f4a7e64f5e0096,content_000f61e9d6a1c60f,3,LOW_QUERY_VISIBILITY,QUICK_WIN
8,client_810019792c9b8efc,content_0010b207b9fd5b52,3,LOW_QUERY_VISIBILITY,QUICK_WIN
9,client_73cda7b4e4f265ea,content_0010bb53cf96978d,3,LOW_QUERY_VISIBILITY,QUICK_WIN


In [17]:
import os

os.makedirs('work/outputs', exist_ok=True)

In [20]:
output_path = 'work/outputs/baseline_action_score.csv'

baseline_queue.to_csv(output_path, index=False)

print(f'Baseline queue written to: {output_path}')
print(f'Rows: {len(baseline_queue):,}')

Baseline queue written to: work/outputs/baseline_action_score.csv
Rows: 102,203


## 3. Top-10 review

| Rank | Action | Why it is here | What would make it wrong |
|---|---|---|---|
| 1 | QUICK_WIN | Only 1 visible query. | Very low impression volume and 100% concentration in one query may indicate limited demand rather than a quick win. |
| 2 | QUICK_WIN | Only 3 visible queries. | Very low recent impressions and position 68.5 may indicate a deeper visibility problem rather than a quick win. |
| 3 | QUICK_WIN | Only 1 visible query. | All impressions are concentrated in one query and kept impressions are very low, so the opportunity may be too narrow. |
| 4 | QUICK_WIN | Only 3 visible queries. | Low ranking position and declining impressions may require a deeper ranking intervention rather than a quick win. |
| 5 | QUICK_WIN | Only 1 visible query. | 100% query concentration may limit the opportunity even though the page has a relatively good position. |
| 6 | QUICK_WIN | Only 1 visible query. | Low impression volume and single-query concentration may indicate limited demand. |
| 7 | QUICK_WIN | Only 3 visible queries. | Low ranking position and declining impressions may indicate a deeper problem than query visibility alone. |
| 8 | QUICK_WIN | Only 2 visible queries. | Low query diversity may not represent a real opportunity despite the declining impressions. |
| 9 | QUICK_WIN | Only 1 visible query. | The page already ranks well, while impression volume is very low, so intervention may not produce a meaningful quick win. |
| 10 | QUICK_WIN | Only 1 visible query. | 100% query concentration and declining impressions may indicate concentrated demand rather than an actionable quick win. |

In [21]:
top10 = baseline_queue.head(10).copy()

top10


,client_hash_id,content_hash_id,score,reason_code,action_label
0,client_7de9989c909e91a5,content_00000c99413ae2ad,3,LOW_QUERY_VISIBILITY,QUICK_WIN
1,client_9958f0a7ae1df715,content_0002bd310bf01f15,3,LOW_QUERY_VISIBILITY,QUICK_WIN
2,client_fef1a8f436438636,content_00032be2df0005ca,3,LOW_QUERY_VISIBILITY,QUICK_WIN
3,client_73cda7b4e4f265ea,content_00033c286cc93446,3,LOW_QUERY_VISIBILITY,QUICK_WIN
4,client_157ffe4d4a595515,content_00039f4c7a954114,3,LOW_QUERY_VISIBILITY,QUICK_WIN
5,client_73cda7b4e4f265ea,content_0009df3c2b96f209,3,LOW_QUERY_VISIBILITY,QUICK_WIN
6,client_e5c2aa26a8598242,content_000b09c2a4ef4ba8,3,LOW_QUERY_VISIBILITY,QUICK_WIN
7,client_62f4a7e64f5e0096,content_000f61e9d6a1c60f,3,LOW_QUERY_VISIBILITY,QUICK_WIN
8,client_810019792c9b8efc,content_0010b207b9fd5b52,3,LOW_QUERY_VISIBILITY,QUICK_WIN
9,client_73cda7b4e4f265ea,content_0010bb53cf96978d,3,LOW_QUERY_VISIBILITY,QUICK_WIN


In [22]:
top10_details = model_data[
    model_data['content_hash_id'].isin(top10['content_hash_id'])
][[
    'client_hash_id',
    'content_hash_id',
    'imp_last30',
    'imp_prev30',
    'clk_last30',
    'pos_last30',
    'visible_queries',
    'rare_share',
    'anon_share',
    'top_query_impressions',
    'kept_impressions',
    'top_query_share'
]].copy()

top10_details.sort_values('content_hash_id')

,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
53811,client_7de9989c909e91a5,content_00000c99413ae2ad,333.0,112.0,0.0,7.349937,1.0,0.033708,0.941573,11.0,11.0,1.000000
50317,client_9958f0a7ae1df715,content_0002bd310bf01f15,2.0,132.0,0.0,68.500000,3.0,0.516981,0.056604,72.0,113.0,0.637168
32520,client_fef1a8f436438636,content_00032be2df0005ca,128.0,206.0,0.0,15.686014,1.0,0.030083,0.958506,11.0,11.0,1.000000
82141,client_73cda7b4e4f265ea,content_00033c286cc93446,78.0,196.0,0.0,31.806250,3.0,0.118321,0.673664,69.0,109.0,0.633028
99007,client_157ffe4d4a595515,content_00039f4c7a954114,121.0,108.0,1.0,6.130998,1.0,0.015244,0.902439,27.0,27.0,1.000000
37924,client_73cda7b4e4f265ea,content_0009df3c2b96f209,166.0,122.0,2.0,4.581528,1.0,0.173848,0.744428,55.0,55.0,1.000000
108013,client_e5c2aa26a8598242,content_000b09c2a4ef4ba8,128.0,251.0,1.0,35.340432,3.0,0.125714,0.741429,55.0,93.0,0.591398
31078,client_62f4a7e64f5e0096,content_000f61e9d6a1c60f,109.0,184.0,1.0,15.386371,2.0,0.074394,0.847751,33.0,45.0,0.733333
78953,client_810019792c9b8efc,content_0010b207b9fd5b52,17.0,116.0,0.0,4.064103,1.0,0.028037,0.911215,13.0,13.0,1.000000
83095,client_73cda7b4e4f265ea,content_0010bb53cf96978d,62.0,175.0,0.0,21.344482,1.0,0.151934,0.806630,15.0,15.0,1.000000


## 4. Weak picks

The rule can over-prioritize content with very low query visibility. For example, these weak picks have only one visible query but zero impressions in the last 30 days and no observed recent position. This means that low query visibility alone is not enough to establish a true quick-win opportunity. The baseline may therefore prioritize content with insufficient current visibility to support an actionable intervention.

In [23]:
weak_picks = (
    baseline[
        ['client_hash_id', 'content_hash_id',
         'score', 'action_label',
         'visible_queries', 'imp_last30',
         'imp_prev30', 'pos_last30']
    ]
    .query("score == 3")
    .sort_values(
        by=['imp_last30', 'visible_queries'],
        ascending=[True, True]
    )
    .head(5)
)

weak_picks

,client_hash_id,content_hash_id,score,action_label,visible_queries,imp_last30,imp_prev30,pos_last30
3594,client_e547b89c05043229,content_f8221ed4b63c47d3,3,QUICK_WIN,1.0,0.0,425.0,NaN
3597,client_e547b89c05043229,content_5d6615a94d86f7dd,3,QUICK_WIN,1.0,0.0,129.0,NaN
4331,client_73cda7b4e4f265ea,content_410d20114acabb03,3,QUICK_WIN,1.0,0.0,171.0,NaN
9653,client_08a6a72ff48e62c0,content_6a698428694673fc,3,QUICK_WIN,1.0,0.0,4620.0,NaN
9915,client_08a6a72ff48e62c0,content_1fac5325a6c3576e,3,QUICK_WIN,1.0,0.0,4618.0,NaN


## 5. Self-check

- [x] Two signal checks completed with visible bucket tables and n.
- [x] At least one signal is linked to a real FlyRank flag: `visible_queries` / volume / quick-win.
- [x] Signal verdicts recorded: `imp_prev30` = MIXED; `visible_queries` = OPPOSITE.
- [x] One baseline rule implemented with a score, one reason code, and an action label.
- [x] `is_declining` was used only to validate the signals and was not used by the baseline rule.
- [x] The baseline score uses only `visible_queries`, with no label-derived input.
- [x] No future-window data is used by the baseline rule.
- [x] Ranked queue contains 102,203 rows.
- [x] Queue written to `work/outputs/baseline_action_score.csv`.
- [x] Top-10 review completed with action, rationale, and what could make each pick wrong.
- [x] Weak picks reviewed to identify limitations of the baseline.

In [24]:
print("Columns:", baseline_queue.columns.tolist())
print("Rows:", len(baseline_queue))
print("Scores:", sorted(baseline_queue['score'].unique()))
print("Reason codes:", baseline_queue['reason_code'].unique())
print("Actions:", baseline_queue['action_label'].value_counts().to_dict())

Columns: ['client_hash_id', 'content_hash_id', 'score', 'reason_code', 'action_label']
Rows: 102203
Scores: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
Reason codes: ['LOW_QUERY_VISIBILITY']
Actions: {'QUICK_WIN': 29217, 'MONITOR': 25677, 'NO_ACTION': 25180, 'REVIEW': 22129}
